# Kham Pha, Danh Gia va Lam Sach Bang: job_descriptions

Notebook nay tap trung xu ly bang du lieu van ban chi tiet **`job_descriptions`** trong mo hinh Silver Layer:
- Quan he 1-1 voi bang `jobs` thong qua khoa chinh `job_id`.
- Tach rieng khoi bang `jobs` de toi uu hieu nang truy van bang danh muc va phan tich.
- Lam sach the HTML, chuan hoa dau dong, nhan dien ngon ngu (vi, en, vi-en) va tao ma bam SHA-256 (text_hash) phuc vu phat hien noi dung trung lap.

Cac buoc thuc hien:
1. Nap du lieu Bronze va dong bo khoa ngoai `job_id` tu bang `jobs`.
2. Danh gia ty le khuyet thieu, do dai van ban va muc do lan tap the HTML.
3. Xay dung bo ham lam sach text chuyen dung va bo nhan dien ngon ngu.
4. Tao bang `job_descriptions` chuan hoa theo DDL PostgreSQL.
5. Thong ke phan bo ngon ngu, do dai noi dung va preview ket qua.

In [1]:
import os
import re
import json
import hashlib
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cau hinh hien thi pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Moi truong xu ly bang job_descriptions da san sang!')

Moi truong xu ly bang job_descriptions da san sang!


## 1. Nap Du Lieu Nguon va Dong Bo Khoa Ngoai job_id

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

df_listings = pd.DataFrame(bronze_listings)
df_details = pd.DataFrame(bronze_details)

# Inner join theo dung thu tu da tao trong bang 02_jobs.ipynb
df_merged = pd.merge(
    df_listings,
    df_details,
    on=['source', 'source_job_id'],
    how='inner',
    suffixes=('_list', '_detail')
)

# Gan job_id dong bo 100% voi bang jobs (1-indexed)
df_merged['job_id'] = df_merged.index + 1

print(f'Tong so ban ghi dong bo: {len(df_merged):,} tin')
print(f'Danh sach cac truong van ban tiep nhan: responsibilities_raw, requirements_raw, benefits_raw, overview_raw')

Tong so ban ghi dong bo: 2,327 tin
Danh sach cac truong van ban tiep nhan: responsibilities_raw, requirements_raw, benefits_raw, overview_raw


## 2. Kiem Ke Chat Luong Du Lieu Van Ban (Data Quality Profiling)

In [3]:
text_fields = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw', 'required_skills_raw']
quality_metrics = []

for col in text_fields:
    series = df_merged[col].fillna('').astype(str).str.strip()
    empty_cnt = (series == '').sum()
    html_cnt = series.str.contains(r'<[^>]+>', regex=True).sum()
    lengths = series.apply(len)
    quality_metrics.append({
        'Truong du lieu': col,
        'So luong rong': empty_cnt,
        'Ty le day du (%)': round((len(series) - empty_cnt) / len(series) * 100, 2),
        'Chua the HTML': html_cnt,
        'Ty le lan the HTML (%)': round(html_cnt / len(series) * 100, 2),
        'Do dai ky tu trung binh': int(lengths.mean()),
        'Do dai ky tu toi da': int(lengths.max())
    })

df_quality = pd.DataFrame(quality_metrics)
print('--- BANG KIEM KE CHAT LUONG CAC TRUONG VAN BAN ---')
print(df_quality.to_string(index=False))

--- BANG KIEM KE CHAT LUONG CAC TRUONG VAN BAN ---
      Truong du lieu  So luong rong  Ty le day du (%)  Chua the HTML  Ty le lan the HTML (%)  Do dai ky tu trung binh  Do dai ky tu toi da
        overview_raw            994             57.28              1                    0.04                      590                 6956
responsibilities_raw              1             99.96              2                    0.09                      661                 6444
    requirements_raw              7             99.70              1                    0.04                      600                 9517
        benefits_raw              2             99.91             27                    1.16                      544                 4310
 required_skills_raw              0            100.00              0                    0.00                      111                  419


## 3. Xay Dung Bo Ham Lam Sach & Chuan Hoa Van Ban

Cac quy tac lam sach:
1. **Giai ma ky tu HTML**: Chuyen cac ky tu ma hoa nhu `&nbsp;`, `&amp;`, `&lt;`, `&gt;`, `&quot;` ve ky tu thuc.
2. **Xu ly the ngat dong**: Thay the cac the `<br>`, `</p>`, `</li>`, `</div>`, `</h*>` thanh ky tu xuong dong.
3. **Loc bo toan bo the HTML con lai**: Loai bo tat ca the `<...>` con sot lai.
4. **Chuan hoa dau dong (Bullet points)**: Dong nhat cac ky tu bullet (`•`, `*`, `+`, `–`, `-`) thanh `- `.
5. **Loai bo khoang trang thua**: Xoa khoang trang du thua o dau/cuoi moi dong va giam cac dong trong lien tiep.
6. **Nhan dien ngon ngu**: Xac dinh bai dang la Tieng Viet (`vi`), Tieng Anh (`en`), hoac song ngu (`vi-en`).
7. **Tao ma bam SHA-256**: Tao `text_hash` tren toan bo noi dung van ban sach de phat hien trung lap noi dung.

In [4]:
def clean_html_text(text):
    if not text or pd.isna(text):
        return ''
    s = str(text)
    # 1. Unescape HTML entities
    s = s.replace('&nbsp;', ' ').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"')
    # 2. Thay the the ngat khoi bang xuong dong
    s = re.sub(r'<(?:br|/p|/li|/div|/h\d)[^>]*>', '\n', s, flags=re.IGNORECASE)
    # 3. Loai bo cac the HTML con lai
    s = re.sub(r'<[^>]+>', ' ', s)
    # 4. Chuan hoa cac dau gach dau dong (bullet points)
    s = re.sub(r'^[ \t]*[•\*\-\+–][ \t]*', '- ', s, flags=re.MULTILINE)
    # 5. Lam gon khoang trang giua cac tu
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in s.split('\n')]
    # 6. Giu lai cac dong co noi dung, tranh dong trong keo dai
    compact_lines = [line for line in lines if line]
    return '\n'.join(compact_lines)

def detect_language(text):
    if not text or pd.isna(text):
        return 'vi'
    s = str(text).lower()
    # Dau tieng Viet dac trung
    vn_chars = set('àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ')
    vn_words = ['và', 'các', 'của', 'trong', 'với', 'cho', 'khi', 'được', 'người', 'công việc', 'kinh nghiệm', 'yêu cầu', 'quyền lợi', 'kỹ năng']
    en_words = ['and', 'the', 'of', 'in', 'with', 'for', 'when', 'experience', 'requirements', 'skills', 'responsibilities', 'benefits', 'team', 'develop']
    
    vn_char_count = sum(1 for c in s if c in vn_chars)
    vn_word_hits = sum(1 for w in vn_words if re.search(r'\b' + re.escape(w) + r'\b', s))
    en_word_hits = sum(1 for w in en_words if re.search(r'\b' + re.escape(w) + r'\b', s))
    
    if vn_char_count > 10 or vn_word_hits >= 3:
        if en_word_hits >= 4 and vn_word_hits >= 3:
            return 'vi-en'
        return 'vi'
    elif en_word_hits > vn_word_hits:
        return 'en'
    return 'vi'

def compute_sha256(text):
    if not text:
        return None
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

print('Bo ham lam sach van ban va nhan dien ngon ngu da san sang!')

Bo ham lam sach van ban va nhan dien ngon ngu da san sang!


## 4. Khoi Tao Bang job_descriptions Hoan Chinh

Tien hanh lam sach song song cac cot, ghep noi dung mo ta (`overview_raw` + `responsibilities_raw`), tinh toan ngon ngu va ma bam SHA-256.

In [5]:
desc_records = []
now_iso = datetime.now().isoformat()

for idx, row in df_merged.iterrows():
    job_id = row['job_id']
    
    # Lay du lieu tho
    overview = str(row.get('overview_raw', '') or '').strip()
    responsibilities = str(row.get('responsibilities_raw', '') or '').strip()
    requirements = str(row.get('requirements_raw', '') or '').strip()
    benefits = str(row.get('benefits_raw', '') or '').strip()
    
    # Ghep mo ta cong viec (overview + responsibilities neu co ca hai)
    if overview and responsibilities:
        desc_raw = f"{overview}\n\n{responsibilities}"
    elif overview:
        desc_raw = overview
    else:
        desc_raw = responsibilities
        
    # Lam sach
    desc_clean = clean_html_text(desc_raw)
    req_clean = clean_html_text(requirements)
    ben_clean = clean_html_text(benefits)
    
    # Ket hop toan bo noi dung sach de nhan dien ngon ngu va tao ma bam
    full_content = f"{desc_clean}\n{req_clean}\n{ben_clean}".strip()
    lang = detect_language(full_content)
    text_hash = compute_sha256(full_content)
    
    desc_records.append({
        'job_id': job_id,
        'description_raw': desc_raw,
        'requirements_raw': requirements,
        'benefits_raw': benefits,
        'description_clean': desc_clean,
        'requirements_clean': req_clean,
        'benefits_clean': ben_clean,
        'language': lang,
        'text_hash': text_hash,
        'created_at': now_iso,
        'updated_at': now_iso
    })

df_job_descriptions = pd.DataFrame(desc_records)

print('Tao bang job_descriptions thanh cong:')
print(f'  - Tong so ban ghi: {len(df_job_descriptions):,}')
print(f'  - Tinh toan ven khoa chinh job_id: {df_job_descriptions["job_id"].nunique():,} / {len(df_job_descriptions)} (100.0% duy nhat)')
print(f'  - So ban ghi co ma bam SHA-256 hop le: {df_job_descriptions["text_hash"].notna().sum():,} / {len(df_job_descriptions)}')

Tao bang job_descriptions thanh cong:
  - Tong so ban ghi: 2,327
  - Tinh toan ven khoa chinh job_id: 2,327 / 2327 (100.0% duy nhat)
  - So ban ghi co ma bam SHA-256 hop le: 2,327 / 2327


## 5. Danh Gia Ket Qua, Thong Ke Phan Bo & Preview Bang job_descriptions

In [6]:
print('--- 1. CAU TRUC BANG job_descriptions (INFO) ---')
print(df_job_descriptions.info())

print('\n--- 2. PHAN BO NGON NGU BAI VIET (language) ---')
lang_counts = df_job_descriptions['language'].value_counts()
lang_percent = df_job_descriptions['language'].value_counts(normalize=True) * 100
df_lang_stat = pd.DataFrame({'So luong': lang_counts, 'Ty le (%)': lang_percent.round(2)})
print(df_lang_stat.to_string())

print('\n--- 3. THONG KE SO LUONG TU & DO DAI VAN BAN SAU LAM SACH ---')
df_len_stat = pd.DataFrame({
    'Tu Mo ta': df_job_descriptions['description_clean'].apply(lambda x: len(x.split()) if x else 0),
    'Tu Yeu cau': df_job_descriptions['requirements_clean'].apply(lambda x: len(x.split()) if x else 0),
    'Tu Quyen loi': df_job_descriptions['benefits_clean'].apply(lambda x: len(x.split()) if x else 0)
})
print(df_len_stat.describe().round(1).to_string())

print('\n--- 4. PHAT HIEN TRUNG LAP NOI DUNG THUC TE (text_hash) ---')
unique_hashes = df_job_descriptions['text_hash'].nunique()
duplicate_hashes = len(df_job_descriptions) - unique_hashes
print(f'So ma bam noi dung duy nhat: {unique_hashes:,}')
print(f'So ban ghi trung hoan toan noi dung JD voi tin khac: {duplicate_hashes:,} tin ({duplicate_hashes/len(df_job_descriptions)*100:.2f}%)')

print('\n--- 5. PREVIEW 5 DONG DAU BANG job_descriptions (TRICH DOAN NGAN) ---')
preview_df = df_job_descriptions[['job_id', 'language', 'text_hash', 'description_clean', 'requirements_clean']].copy()
preview_df['description_clean'] = preview_df['description_clean'].apply(lambda x: (x[:80] + '...') if len(x) > 80 else x)
preview_df['requirements_clean'] = preview_df['requirements_clean'].apply(lambda x: (x[:80] + '...') if len(x) > 80 else x)
preview_df['text_hash'] = preview_df['text_hash'].apply(lambda x: (x[:12] + '...') if x else '')
print(preview_df.head(5).to_string(index=False))

--- 1. CAU TRUC BANG job_descriptions (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              2327 non-null   int64
 1   description_raw     2327 non-null   str  
 2   requirements_raw    2327 non-null   str  
 3   benefits_raw        2327 non-null   str  
 4   description_clean   2327 non-null   str  
 5   requirements_clean  2327 non-null   str  
 6   benefits_clean      2327 non-null   str  
 7   language            2327 non-null   str  
 8   text_hash           2327 non-null   str  
 9   created_at          2327 non-null   str  
 10  updated_at          2327 non-null   str  
dtypes: int64(1), str(10)
memory usage: 13.2 MB
None

--- 2. PHAN BO NGON NGU BAI VIET (language) ---
          So luong  Ty le (%)
language                     
vi            1733      74.47
en             328      14.10
vi-en          266    